# 02 · Landmark redesign (Level 2)

Rebuilds the structured features and ClinicalBERT text **so the window length no longer depends on the outcome**.

**The three rules (landmark design):**
1. Pick a fixed moment, the *landmark* `L` (default 12 h after ICU admission).
2. Keep only patients who are still in the ICU **and** haven't had respiratory failure yet at `L`.
3. Everyone gets the same data window: labs from −6 h to `L`, medications from 0 h to `L`.
   New question: **does respiratory failure happen between `L` and 48 h?**

What changed vs. `GRIDS_FeatureEng_BERT_train.ipynb`:

| Original | Here |
|---|---|
| cutoff = `rf_time` for positives, 12 h for negatives | cutoff = `L` for everyone |
| label = failure anywhere in 0–48 h | label = failure in (`L`, 48 h] |
| all stays | only stays still at risk at `L` |
| features incl. `feature_window_hours`, `obs_window_hours`, `is_short_stay`, `stay_duration_hours` | removed (constant, or future information) |
| text says *"Clinical data available for X hours…"* | sentence removed |
| per-stay Python loop (hours) | vectorised pandas (minutes) |

Everything else — the 20 labs × 8 statistics, 13 prescription features, 70/15/15 patient-level split with `random_state=42`, the text template — is kept identical so results are comparable.

**Inputs** (the `data/comb` Drive folder): `cohort.csv`, `labs.csv`, `prescriptions.csv`, `respiratory_failure_labels.csv`
**Outputs** (per landmark): `train_df.csv`, `val_df.csv`, `test_df.csv` with a `clinical_text` column — drop-in replacements for `ClinicalBERT_Train.ipynb` — plus `landmark_results.json`.

⚠️ Outputs go to your Drive, never into the repo. Don't print patient rows.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────
import sys, json, time, re
from pathlib import Path
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ✏️ EDIT THESE
DATA_DIR = Path("/content/drive/MyDrive/USC/ICU-MM-main/ICU-MM/data/comb")   # folder with cohort.csv, labs.csv, ...
SAVE_DIR = Path("/content/drive/MyDrive/USC/ICU-MM-landmark")                 # new folder — doesn't overwrite the team's outputs

LANDMARKS_H  = [12]      # try [3, 6, 12] to see the lead-time vs. sample-size trade-off
HORIZON_H    = 48        # label window end (the label script only looked 48 h ahead)
LAB_START_H  = -6        # same as original WINDOW_START_HOURS (ED labs before ICU admission)
SEED         = 42

SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Same constants as the original notebook
TOP_LABS = [
    "Glucose", "Potassium", "Sodium", "Chloride",
    "Hemoglobin", "Creatinine", "Urea Nitrogen", "Bicarbonate",
    "Hematocrit", "Anion Gap", "Magnesium", "Platelet Count",
    "White Blood Cells", "MCHC", "Red Blood Cells", "MCV",
    "MCH", "RDW", "Phosphate", "Calcium, Total",
]
DRUG_CATEGORIES = {
    "antibiotic":    ["vancomycin", "piperacillin", "cefepime", "meropenem", "ciprofloxacin",
                      "metronidazole", "levofloxacin", "ampicillin", "ceftriaxone", "azithromycin"],
    "sedation":      ["propofol", "midazolam", "lorazepam", "dexmedetomidine", "ketamine", "diazepam"],
    "opioid":        ["fentanyl", "morphine", "hydromorphone", "dilaudid", "oxycodone", "methadone", "remifentanil"],
    "cardiac":       ["metoprolol", "amiodarone", "digoxin", "diltiazem", "lisinopril", "carvedilol", "atenolol", "labetalol"],
    "anticoagulant": ["heparin", "warfarin", "enoxaparin", "apixaban", "rivaroxaban", "fondaparinux"],
    "insulin":       ["insulin"],
    "diuretic":      ["furosemide", "torsemide", "bumetanide", "spironolactone"],
    "steroid":       ["hydrocortisone", "methylprednisolone", "dexamethasone", "prednisone", "fludrocortisone"],
}
IV_ROUTES     = {"iv", "iv drip", "iv bolus"}
SERIOUS_ROUTE = "iv drip"
safe = lambda lab: lab.replace(" ", "_").replace(",", "")

In [ ]:
# ── Cohort: same cleaning as the original (cell 5) ────────────
cohort = pd.read_csv(DATA_DIR / "cohort.csv", parse_dates=["icu_intime", "icu_outtime"])
cohort["hadm_id"] = cohort["hadm_id"].astype("int64")

# Remove bouncebacks (gap < 24h between consecutive stays in one admission) — unchanged
stays_per_adm = cohort.groupby("hadm_id")["stay_id"].nunique()
multi = cohort[cohort["hadm_id"].isin(stays_per_adm[stays_per_adm > 1].index)].sort_values(["hadm_id", "icu_intime"]).copy()
multi["prev_out"] = multi.groupby("hadm_id")["icu_outtime"].shift(1)
bounce = multi.loc[(multi["icu_intime"] - multi["prev_out"]).dt.total_seconds() / 3600 < 24, "stay_id"]
cohort = cohort[~cohort["stay_id"].isin(bounce)].reset_index(drop=True)

labels = pd.read_csv(DATA_DIR / "respiratory_failure_labels.csv", parse_dates=["rf_time"])
cohort = cohort.merge(labels[["stay_id", "respiratory_failure", "rf_time"]], on="stay_id", how="left")
cohort["respiratory_failure"] = cohort["respiratory_failure"].fillna(0).astype(int)
cohort["rf_hours"] = (cohort["rf_time"] - cohort["icu_intime"]).dt.total_seconds() / 3600
cohort["stay_hours"] = (cohort["icu_outtime"] - cohort["icu_intime"]).dt.total_seconds() / 3600

print(f"Clean cohort: {len(cohort):,} stays | original-style label positive: {cohort['respiratory_failure'].sum():,} "
      f"({cohort['respiratory_failure'].mean()*100:.1f}%) | median time to RF {cohort['rf_hours'].median():.1f} h")

# How many positives survive each landmark?
print("\nPositives still predictable at each landmark (RF after L, within 48h):")
for L in [3, 6, 12, 24]:
    print(f"  L = {L:2d}h → {((cohort['rf_hours'] > L) & (cohort['rf_hours'] <= HORIZON_H)).sum():>6,}")

In [ ]:
# ── Labs: read once, in chunks, keep only rows inside [-6h, max L] ──
MAX_L = max(LANDMARKS_H)
lab_long_path = SAVE_DIR / f"_labs_long_upto_{MAX_L}h.parquet"

if lab_long_path.exists():
    lab_long = pd.read_parquet(lab_long_path)
    print(f"Loaded cached lab rows: {len(lab_long):,}")
else:
    t0 = time.time()
    stay_map = cohort[["stay_id", "hadm_id", "icu_intime"]]
    keep_hadm, keep_labs = set(stay_map["hadm_id"]), set(TOP_LABS)
    parts = []
    for i, chunk in enumerate(pd.read_csv(DATA_DIR / "labs.csv", usecols=["hadm_id", "lab_time", "lab_name", "value"],
                                          chunksize=2_000_000)):
        chunk = chunk[chunk["lab_name"].isin(keep_labs) & chunk["hadm_id"].isin(keep_hadm)]
        if chunk.empty:
            continue
        chunk["hadm_id"]  = chunk["hadm_id"].astype("int64")
        chunk["lab_time"] = pd.to_datetime(chunk["lab_time"])
        chunk["value"]    = pd.to_numeric(chunk["value"], errors="coerce")
        chunk = chunk.dropna(subset=["value"]).merge(stay_map, on="hadm_id")   # one lab row → every stay in that admission (same as original)
        chunk["hours"] = (chunk["lab_time"] - chunk["icu_intime"]).dt.total_seconds() / 3600
        chunk = chunk[(chunk["hours"] >= LAB_START_H) & (chunk["hours"] <= MAX_L)]
        parts.append(chunk[["stay_id", "lab_name", "hours", "value"]].astype({"hours": "float32", "value": "float32"}))
        print(f"  chunk {i+1}: kept {sum(len(p) for p in parts):,} rows so far ({time.time()-t0:.0f}s)")
    lab_long = pd.concat(parts, ignore_index=True)
    lab_long.to_parquet(lab_long_path, index=False)
    print(f"Lab rows in window: {len(lab_long):,}  ({time.time()-t0:.0f}s) — cached to Drive")

In [ ]:
# ── Prescriptions: read once ──────────────────────────────────
presc = pd.read_csv(DATA_DIR / "prescriptions.csv", usecols=["hadm_id", "med_starttime", "drug", "route"],
                    parse_dates=["med_starttime"])
presc = presc[presc["hadm_id"].isin(set(cohort["hadm_id"]))].copy()
presc["hadm_id"] = presc["hadm_id"].astype("int64")
presc = presc.merge(cohort[["stay_id", "hadm_id", "icu_intime"]], on="hadm_id")
presc["hours"] = (presc["med_starttime"] - presc["icu_intime"]).dt.total_seconds() / 3600
presc = presc[(presc["hours"] >= 0) & (presc["hours"] <= MAX_L)]
presc["drug_l"]  = presc["drug"].astype(str).str.lower()
presc["route_l"] = presc["route"].astype(str).str.lower()
for cat, kws in DRUG_CATEGORIES.items():
    presc[f"has_{cat}"] = presc["drug_l"].str.contains("|".join(map(re.escape, kws)), regex=True).astype(int)
presc["is_iv"]   = presc["route_l"].isin(IV_ROUTES).astype(int)
presc["is_drip"] = (presc["route_l"] == SERIOUS_ROUTE).astype(int)
print(f"Prescription rows in [0, {MAX_L}h]: {len(presc):,}")

In [ ]:
# ── Feature builders (same 160 lab + 13 prescription features as the original) ──
LAB_STATS = ["mean", "min", "max", "count", "first", "last", "delta", "early_mean"]

def lab_features(stay_ids, L):
    d = lab_long[(lab_long["hours"] <= L) & lab_long["stay_id"].isin(stay_ids)].sort_values(["stay_id", "lab_name", "hours"])
    g = d.groupby(["stay_id", "lab_name"])["value"]
    agg = g.agg(["mean", "min", "max", "count", "first", "last"])
    agg["delta"] = agg["last"] - agg["first"]
    early = d[(d["hours"] >= 0) & (d["hours"] <= 6)].groupby(["stay_id", "lab_name"])["value"].mean().rename("early_mean")
    agg = agg.join(early)
    wide = agg.unstack("lab_name")                      # columns: (stat, lab)
    wide.columns = [f"{safe(lab)}_{stat}" for stat, lab in wide.columns]
    wide = wide.reindex(index=pd.Index(stay_ids, name="stay_id"),
                        columns=[f"{safe(l)}_{s}" for l in TOP_LABS for s in LAB_STATS])
    count_cols = [c for c in wide.columns if c.endswith("_count")]
    wide[count_cols] = wide[count_cols].fillna(0).astype(int)   # original: count=0, others NaN when no data
    return wide.reset_index()

def presc_features(stay_ids, L):
    d = presc[(presc["hours"] <= L) & presc["stay_id"].isin(stay_ids)]
    g = d.groupby("stay_id")
    out = pd.DataFrame({
        "total_presc":   g.size(),
        "unique_drugs":  g["drug"].nunique(),
        "iv_count":      g["is_iv"].sum(),
        "iv_drip_count": g["is_drip"].sum(),
    })
    out["iv_drip_ratio"] = out["iv_drip_count"] / out["total_presc"]
    for cat in DRUG_CATEGORIES:
        out[f"has_{cat}"] = g[f"has_{cat}"].max()
    out = out.reindex(pd.Index(stay_ids, name="stay_id")).fillna(0)
    return out.reset_index()

In [ ]:
# ── Text serialisation — identical to the original EXCEPT the window sentence is gone ──
LAB_GROUPS = {
    "Metabolic panel": [("Glucose", "Glucose", "mg/dL"), ("Sodium", "Sodium", "mEq/L"), ("Potassium", "Potassium", "mEq/L"),
                        ("Chloride", "Chloride", "mEq/L"), ("Bicarbonate", "Bicarbonate", "mEq/L"), ("Anion_Gap", "Anion Gap", "mEq/L"),
                        ("Calcium_Total", "Calcium", "mg/dL"), ("Magnesium", "Magnesium", "mg/dL"), ("Phosphate", "Phosphate", "mg/dL")],
    "Renal function":  [("Creatinine", "Creatinine", "mg/dL"), ("Urea_Nitrogen", "BUN", "mg/dL")],
    "Blood count":     [("Hemoglobin", "Hemoglobin", "g/dL"), ("Hematocrit", "Hematocrit", "%"), ("White_Blood_Cells", "WBC", "K/uL"),
                        ("Platelet_Count", "Platelets", "K/uL"), ("Red_Blood_Cells", "RBC", "M/uL"), ("MCHC", "MCHC", "g/dL"),
                        ("MCV", "MCV", "fL"), ("MCH", "MCH", "pg"), ("RDW", "RDW", "%")],
}
CATEGORY_DISPLAY = {"antibiotic": "antibiotics", "sedation": "sedatives", "opioid": "opioids", "cardiac": "cardiac medications",
                    "anticoagulant": "anticoagulation", "insulin": "insulin", "diuretic": "diuretics", "steroid": "corticosteroids"}

def row_to_clinical_text(row):
    parts = [f"Patient is a {int(row['anchor_age'])}-year-old {'male' if row['sex'] == 1 else 'female'}."]
    # (removed) "Clinical data available for X hours following ICU admission."  ← this sentence was the leak
    for group_name, labs in LAB_GROUPS.items():
        mentions = []
        for s, disp, unit in labs:
            mean_val, delta_val, count_val = row.get(f"{s}_mean"), row.get(f"{s}_delta"), row.get(f"{s}_count", 0)
            if pd.isna(mean_val) or count_val == 0:
                continue
            m = f"{disp} {mean_val:.1f} {unit}"
            if not pd.isna(delta_val) and count_val > 1:
                m += " (rising)" if delta_val > 0 else (" (falling)" if delta_val < 0 else "")
            mentions.append(m)
        if mentions:
            parts.append(f"{group_name}: {', '.join(mentions)}.")
    n_presc, n_unique, n_drip = int(row["total_presc"]), int(row["unique_drugs"]), int(row["iv_drip_count"])
    if n_presc == 0:
        parts.append("No medications ordered during observation window.")
    else:
        parts.append(f"Received {n_presc} medication orders ({n_unique} unique drugs) during observation window.")
        if n_drip > 0:
            parts.append(f"{n_drip} continuous IV {'drip' if n_drip == 1 else 'drips'} running.")
    active = [disp for cat, disp in CATEGORY_DISPLAY.items() if row.get(f"has_{cat}", 0) == 1]
    if active:
        parts.append(f"Active medications include: {', '.join(active)}.")
    return " ".join(parts)

In [ ]:
# ── Build one landmark dataset ────────────────────────────────
from sklearn.model_selection import GroupShuffleSplit

def build_landmark(L):
    at_risk = (cohort["stay_hours"] > L) & (cohort["rf_hours"].isna() | (cohort["rf_hours"] > L))   # rule 2
    c = cohort[at_risk].copy()
    c["respiratory_failure"] = ((c["rf_hours"] > L) & (c["rf_hours"] <= HORIZON_H)).astype(int)     # new label
    # note: no feature_window_hours / obs_window_hours / is_short_stay / stay_duration_hours
    admin = pd.DataFrame({"stay_id": c["stay_id"].values, "subject_id": c["subject_id"].values,
                          "anchor_age": c["anchor_age"].values, "sex": (c["sex"] == "M").astype(int).values,
                          "respiratory_failure": c["respiratory_failure"].values})
    ids = admin["stay_id"].tolist()
    df = admin.merge(lab_features(ids, L), on="stay_id", how="left").merge(presc_features(ids, L), on="stay_id", how="left")
    df["clinical_text"] = df.apply(row_to_clinical_text, axis=1)

    # Same 70/15/15 patient-level split as the original
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    tr_idx, tmp_idx = next(gss1.split(df, groups=df["subject_id"]))
    train, tmp = df.iloc[tr_idx].reset_index(drop=True), df.iloc[tmp_idx].reset_index(drop=True)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    va_idx, te_idx = next(gss2.split(tmp, groups=tmp["subject_id"]))
    val, test = tmp.iloc[va_idx].reset_index(drop=True), tmp.iloc[te_idx].reset_index(drop=True)
    assert not (set(train.subject_id) & set(val.subject_id)) and not (set(train.subject_id) & set(test.subject_id)) \
        and not (set(val.subject_id) & set(test.subject_id)), "patient overlap between splits!"
    return train, val, test

In [ ]:
# ── Structured baselines on the honest data ───────────────────
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def evaluate(train, val, test):
    feats = [c for c in train.columns if c not in {"stay_id", "subject_id", "respiratory_failure", "clinical_text"}]
    models = {
        "LogReg": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                LogisticRegression(max_iter=3000, class_weight="balanced")),
        "GradBoost": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, class_weight="balanced", random_state=SEED),
    }
    out = []
    for name, m in models.items():
        m.fit(train[feats], train["respiratory_failure"])
        for split_name, d in [("val", val), ("test", test)]:
            p = m.predict_proba(d[feats])[:, 1]
            out.append({"model": name, "split": split_name,
                        "AUROC": round(roc_auc_score(d["respiratory_failure"], p), 4),
                        "AUPRC": round(average_precision_score(d["respiratory_failure"], p), 4),
                        "AUPRC_chance": round(d["respiratory_failure"].mean(), 4)})
    return out, len(feats)

In [ ]:
# ── Run ───────────────────────────────────────────────────────
all_results = {}
for L in LANDMARKS_H:
    t0 = time.time()
    train, val, test = build_landmark(L)
    out_dir = SAVE_DIR / f"landmark_{L}h"
    out_dir.mkdir(parents=True, exist_ok=True)
    train.to_csv(out_dir / "train_df.csv", index=False)
    val.to_csv(out_dir / "val_df.csv", index=False)
    test.to_csv(out_dir / "test_df.csv", index=False)

    metrics, n_feats = evaluate(train, val, test)
    all_results[f"{L}h"] = {
        "n_train": len(train), "n_val": len(val), "n_test": len(test),
        "n_positive_total": int(train.respiratory_failure.sum() + val.respiratory_failure.sum() + test.respiratory_failure.sum()),
        "positive_rate_train": round(float(train.respiratory_failure.mean()), 4),
        "n_features": n_feats, "metrics": metrics,
    }
    print(f"\n=== Landmark {L}h  ({time.time()-t0:.0f}s) ===")
    print(f"train {len(train):,} | val {len(val):,} | test {len(test):,} | positives {all_results[f'{L}h']['n_positive_total']:,} "
          f"| positive rate {train.respiratory_failure.mean()*100:.1f}%")
    print(pd.DataFrame(metrics).to_string(index=False))
    print(f"→ files for ClinicalBERT: {out_dir}")

with open(SAVE_DIR / "landmark_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nSaved results → {SAVE_DIR / 'landmark_results.json'}  (copy this file into the repo's results/ folder)")

## Next: retrain ClinicalBERT on the honest data

In `ClinicalBERT_Train.ipynb`, change only the three `pd.read_csv(...)` paths in cell 2 to
`SAVE_DIR / "landmark_12h" / "train_df.csv"` (and `val_df.csv`, `test_df.csv`). Nothing else needs to change —
column names are the same.

**What to expect:** AUROC somewhere in the 0.70s–low 0.80s and a positive rate around 5–10 %. At that imbalance,
report **AUPRC** next to AUROC and compare it to `AUPRC_chance` (the positive rate) — that's the honest yardstick.